# MIMIC-IV v3.1 cohort and first-24-hour features

Build the MIMIC-IV (v3.1) first-24-hour ICU cohort used for temporal validation.

Runs locally on the credentialed data; nothing is uploaded anywhere.

    python mimic/extract_mimic.py --root "D:/MIMIC_IV/mimic_iv/mimic-iv-3.1" --out mimic_cohort.csv.gz

Cohort: first ICU stay per patient, age >= 18 at ICU admission, ICU length of
stay >= 24 h, alive at 24 h after ICU admission. Outcome: in-hospital death
(admissions.hospital_expire_flag). Features are aggregated over the first 24 h
of the ICU stay (vitals) or from 6 h before to 24 h after ICU admission (labs).
Every feature is defined by a fixed itemid list (FEATURE_MAP below), so the same
definition is applied identically to the development era (anchor_year_group
2008-2016) and the temporal validation era (2017-2022).

## Setup
Loads `00_config` (paths) and the library notebooks from this folder.

In [10]:
# Load other notebooks in this folder as modules (all notebooks live in one folder).
import os, sys, types
import nbformat

NB_DIR = os.getcwd()   # open Jupyter in the notebooks folder


def import_notebook(name, filename):
    """Execute the code cells of a notebook in this folder into a module named `name`."""
    if name in sys.modules:
        return sys.modules[name]
    path = os.path.join(NB_DIR, filename)
    mod = types.ModuleType(name)
    mod.__file__ = path
    sys.modules[name] = mod
    for cell in nbformat.read(path, as_version=4).cells:
        if cell.cell_type == "code":
            exec(compile(cell.source, path, "exec"), mod.__dict__)
    return mod


config = import_notebook("config", "00_config.ipynb")

In [11]:
import argparse
import os
import time
import duckdb
import pandas as pd

## Feature dictionary

In [12]:
# ------------------------------------------------------------------ feature dictionary
# name: (source table, itemids, aggregation, plausible range)
VITALS = {
    "hr":       ([220045], (1, 300)),
    "sbp":      ([220050, 220179], (20, 300)),
    "map":      ([220052, 220181, 225312], (10, 250)),
    "rr":       ([220210, 224690], (1, 80)),
    "spo2":     ([220277], (40, 100)),
    "temp_c":   ([223762], (25, 45)),
    "temp_f":   ([223761], (77, 113)),
    "gcs_eye":  ([220739], (1, 4)),
    "gcs_verbal": ([223900], (1, 5)),
    "gcs_motor": ([223901], (1, 6)),
}
VITAL_AGG = {"hr": ["min", "max", "mean"], "sbp": ["min", "mean"], "map": ["min", "mean"],
             "rr": ["max", "mean"], "spo2": ["min", "mean"], "temp": ["min", "max"]}
LABS = {
    "creatinine": ([50912], "max", (0.1, 30)),
    "bun":        ([51006], "max", (1, 300)),
    "sodium_min": ([50983], "min", (100, 190)),
    "sodium_max": ([50983], "max", (100, 190)),
    "potassium":  ([50971], "max", (1, 12)),
    "bicarbonate": ([50882], "min", (2, 60)),
    "glucose":    ([50931], "max", (10, 2000)),
    "hemoglobin": ([51222], "min", (2, 25)),
    "wbc":        ([51301, 51300], "max", (0.1, 500)),
    "platelets":  ([51265], "min", (1, 2000)),
    "lactate":    ([50813], "max", (0.1, 30)),
    "bilirubin":  ([50885], "max", (0.1, 70)),
    "albumin":    ([50862], "min", (0.5, 7)),
    "inr":        ([51237], "max", (0.5, 20)),
    "aniongap":   ([50868], "max", (1, 60)),
}
VENT_ITEMIDS = [225792]  # procedureevents: invasive ventilation
FEATURE_MAP = dict(vitals={k: v[0] for k, v in VITALS.items()}, labs={k: v[0] for k, v in LABS.items()},
                   ventilation=VENT_ITEMIDS)

### `p` (function)

In [13]:
def p(root, *parts):
    for ext in (".csv.gz", ".csv", ".parquet"):
        f = os.path.join(root, *parts) + ext
        if os.path.exists(f):
            return f.replace("\\", "/")
    raise FileNotFoundError(os.path.join(root, *parts))

### `reader` (function)

In [14]:
def reader(f):
    return f"read_parquet('{f}')" if f.endswith(".parquet") else f"read_csv_auto('{f}', header=true)"

### `build` (function)

In [15]:
def build(root, threads=4):
    root = config.find_mimic_root(root)  # folder containing hosp/ and icu/
    print("MIMIC-IV files:", root)
    con = duckdb.connect()
    con.execute(f"PRAGMA threads={threads}")
    T = {t: reader(p(root, *t.split("/"))) for t in
         ["hosp/patients", "hosp/admissions", "icu/icustays", "icu/chartevents", "hosp/labevents",
          "icu/procedureevents"]}
    t0 = time.time()
    con.execute(f"""
    CREATE TEMP TABLE cohort AS
    WITH s AS (
      SELECT i.subject_id, i.hadm_id, i.stay_id, i.first_careunit, CAST(i.intime AS TIMESTAMP) intime,
             CAST(i.outtime AS TIMESTAMP) outtime, i.los,
             ROW_NUMBER() OVER (PARTITION BY i.subject_id ORDER BY i.intime) rn
      FROM {T['icu/icustays']} i)
    SELECT s.*, pa.gender, pa.anchor_age + (year(s.intime) - pa.anchor_year) AS age, pa.anchor_year_group,
           a.admission_type, a.hospital_expire_flag, CAST(a.deathtime AS TIMESTAMP) deathtime
    FROM s JOIN {T['hosp/patients']} pa USING (subject_id)
           JOIN {T['hosp/admissions']} a ON a.hadm_id = s.hadm_id
    WHERE s.rn = 1
      AND pa.anchor_age + (year(s.intime) - pa.anchor_year) >= 18
      AND s.los >= 1
      AND (a.deathtime IS NULL OR CAST(a.deathtime AS TIMESTAMP) > s.intime + INTERVAL 24 HOUR)
    """)
    n = con.execute("SELECT count(*) FROM cohort").fetchone()[0]
    print(f"cohort: {n} stays ({time.time() - t0:.0f}s)", flush=True)

    # ---------------- vitals (chartevents, first 24 h)
    ids = sorted({i for v in VITALS.values() for i in v[0]})
    case = " ".join(f"WHEN itemid IN ({','.join(map(str, v[0]))}) THEN '{k}'" for k, v in VITALS.items())
    rng = " OR ".join(f"(name='{k}' AND valuenum BETWEEN {v[1][0]} AND {v[1][1]})" for k, v in VITALS.items())
    con.execute(f"""
    CREATE TEMP TABLE vit AS
    SELECT stay_id, name, CASE WHEN name='temp_f' THEN (valuenum-32)/1.8 ELSE valuenum END AS v
    FROM (SELECT c.stay_id, c.valuenum, CASE {case} END AS name
          FROM {T['icu/chartevents']} c JOIN cohort k ON c.stay_id = k.stay_id
          WHERE c.itemid IN ({','.join(map(str, ids))}) AND c.valuenum IS NOT NULL
            AND CAST(c.charttime AS TIMESTAMP) BETWEEN k.intime AND k.intime + INTERVAL 24 HOUR)
    WHERE {rng}
    """)
    con.execute("UPDATE vit SET name='temp' WHERE name IN ('temp_c','temp_f')")
    print(f"vitals extracted ({time.time() - t0:.0f}s)", flush=True)
    sel = []
    for k, aggs in VITAL_AGG.items():
        for a in aggs:
            sel.append(f"{a.upper() if a != 'mean' else 'AVG'}(CASE WHEN name='{k}' THEN v END) AS {k}_{a}")
    sel.append("MIN(CASE WHEN name='gcs_eye' THEN v END) + MIN(CASE WHEN name='gcs_verbal' THEN v END)"
               " + MIN(CASE WHEN name='gcs_motor' THEN v END) AS gcs_min")
    vit = con.execute(f"SELECT stay_id, {', '.join(sel)} FROM vit GROUP BY stay_id").df()

    # ---------------- labs (labevents, -6 h to +24 h; joined on hadm_id)
    lab_ids = sorted({i for v in LABS.values() for i in v[0]})
    lsel = []
    for k, (iids, agg, (lo, hi)) in LABS.items():
        cond = f"itemid IN ({','.join(map(str, iids))}) AND valuenum BETWEEN {lo} AND {hi}"
        lsel.append(f"{agg.upper()}(CASE WHEN {cond} THEN valuenum END) AS {k}")
    lab = con.execute(f"""
    SELECT k.stay_id, {', '.join(lsel)}
    FROM {T['hosp/labevents']} l JOIN cohort k ON l.hadm_id = k.hadm_id
    WHERE l.itemid IN ({','.join(map(str, lab_ids))}) AND l.valuenum IS NOT NULL
      AND CAST(l.charttime AS TIMESTAMP) BETWEEN k.intime - INTERVAL 6 HOUR AND k.intime + INTERVAL 24 HOUR
    GROUP BY k.stay_id""").df()
    print(f"labs extracted ({time.time() - t0:.0f}s)", flush=True)

    # ---------------- invasive ventilation started within first 24 h
    vent = con.execute(f"""
    SELECT DISTINCT k.stay_id, 1 AS vent_24h
    FROM {T['icu/procedureevents']} pe JOIN cohort k ON pe.stay_id = k.stay_id
    WHERE pe.itemid IN ({','.join(map(str, VENT_ITEMIDS))})
      AND CAST(pe.starttime AS TIMESTAMP) <= k.intime + INTERVAL 24 HOUR""").df()

    coh = con.execute("SELECT stay_id, subject_id, age, gender, anchor_year_group, admission_type, "
                      "first_careunit, hospital_expire_flag FROM cohort").df()
    df = coh.merge(vit, on="stay_id", how="left").merge(lab, on="stay_id", how="left") \
            .merge(vent, on="stay_id", how="left")
    df["vent_24h"] = df["vent_24h"].fillna(0).astype(int)
    df["era"] = df["anchor_year_group"].map(lambda g: "2008-2016" if int(str(g)[:4]) <= 2014 else "2017-2022")
    print(f"done: {df.shape} ({time.time() - t0:.0f}s)", flush=True)
    return df

## Run
Reads `chartevents` and `labevents` directly from the compressed files under `MIMIC_ROOT`; expect 30-90 minutes and about 8 GB of RAM. Everything stays on this computer.

In [16]:
THREADS = 4
if __name__ == "__main__":
    df = build(config.MIMIC_ROOT, THREADS)
    df.to_csv(config.MIMIC_COHORT, index=False)
    print("saved", config.MIMIC_COHORT, df.shape)

MIMIC-IV files: D:\MIMIC_IV\mimic_iv\mimic-iv-3.1
cohort: 51667 stays (1s)
vitals extracted (113s)
labs extracted (153s)
done: (51667, 39) (154s)
saved D:\New_Research_2027\Clinical_XAI_Faithfulness\outputs\mimic_cohort.csv.gz (51667, 39)
